<a href="https://colab.research.google.com/github/Leewenct/Hotfix/blob/main/RE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Prepare colab**

In [ ]:
!apt-get update -qq
!apt-get install -y -qq mingw-w64 git
!git clone --depth 1 https://github.com/ocornut/imgui.git
!git clone --depth 1 https://github.com/TsudaKageyu/minhook.git

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
fatal: destination path 'imgui' already exists and is not an empty directory.
fatal: destination path 'minhook' already exists and is not an empty directory.


**Cell 2: Shared**

In [ ]:
%%writefile shared.h
#pragma once

#define WIN32_LEAN_AND_MEAN
#define NOMINMAX

#include <windows.h>
#include <d3d11.h>
#include <dxgi.h>
#include <atomic>

extern std::atomic<bool> g_noCooldownEnabled;
extern std::atomic<bool> g_skillCooldownEnabled;
extern std::atomic<bool> g_ultimateCooldownEnabled;
extern std::atomic<bool> g_dashCooldownEnabled;
extern std::atomic<bool> g_unlimitedEnergyEnabled;
extern std::atomic<bool> g_speedHackEnabled;

void LogError(const char* message);

bool GetD3D11Targets(
    void** presentTarget,
    void** resizeTarget
);

bool InstallOverlayHooks();

bool InstallCooldownHook();

Overwriting shared.h


**Cell 3: Overlay**

In [ ]:
%%writefile overlay.cpp

#include "shared.h"
#include "MinHook.h"

#include "imgui.h"
#include "backends/imgui_impl_win32.h"
#include "backends/imgui_impl_dx11.h"

#include <mutex>

#pragma comment(lib, "d3d11.lib")
#pragma comment(lib, "dxgi.lib")

namespace OverlayConfig
{
    constexpr const char* WindowTitle = "[TAB] to toggle overlay.";
    constexpr const char* FeaturesTabName = "Features";
    constexpr const char* AboutTabName = "About";

    constexpr const char* NoCooldownName = "No Cooldown";
    constexpr const char* SkillName = "Skill";
    constexpr const char* UltimateName = "Ultimate";
    constexpr const char* DashName = "Dash";
    constexpr const char* EnergyName = "Unlimited Energy";
    constexpr const char* SpeedHackName = "Speed Hack";

    constexpr const char* GameName = "Stella Sora";
    constexpr const char* GameVersion = "2.1";

    constexpr int ToggleKey = VK_TAB;
    constexpr bool StartVisible = false;

    constexpr float WindowWidth = 480.0f;
    constexpr float WindowHeight = 280.0f;
    constexpr float SidebarWidth = 120.0f;
    constexpr float FontScale = 1.0f;
}

extern IMGUI_IMPL_API LRESULT ImGui_ImplWin32_WndProcHandler(HWND hWnd, UINT msg, WPARAM wParam, LPARAM lParam);

using PresentFn = HRESULT(STDMETHODCALLTYPE*)(IDXGISwapChain*, UINT, UINT);
using ResizeBuffersFn = HRESULT(STDMETHODCALLTYPE*)(IDXGISwapChain*, UINT, UINT, UINT, DXGI_FORMAT, UINT);

static PresentFn OriginalPresent = nullptr;
static ResizeBuffersFn OriginalResizeBuffers = nullptr;

static ID3D11Device* Device = nullptr;
static ID3D11DeviceContext* Context = nullptr;
static ID3D11RenderTargetView* RenderTarget = nullptr;

static HWND GameWindow = nullptr;
static WNDPROC OriginalWndProc = nullptr;

static bool Initialized = false;
static std::atomic<bool> OverlayVisible{ OverlayConfig::StartVisible };
static int ActiveTab = 0;

static RECT OverlayRect{};
static bool OverlayRectValid = false;

static std::mutex StateLock;
static std::mutex OverlayRectLock;

static LRESULT CALLBACK WindowProc(HWND hWnd, UINT msg, WPARAM wParam, LPARAM lParam);

static void UpdateOverlayRect(const ImVec2& position, const ImVec2& size)
{
    std::lock_guard<std::mutex> lock(OverlayRectLock);
    OverlayRect.left = static_cast<LONG>(position.x);
    OverlayRect.top = static_cast<LONG>(position.y);
    OverlayRect.right = static_cast<LONG>(position.x + size.x);
    OverlayRect.bottom = static_cast<LONG>(position.y + size.y);
    OverlayRectValid = true;
}

static bool IsMouseInsideOverlay(HWND hWnd)
{
    POINT cursor{};
    if (!GetCursorPos(&cursor)) return false;
    ScreenToClient(hWnd, &cursor);
    std::lock_guard<std::mutex> lock(OverlayRectLock);
    return OverlayRectValid && PtInRect(&OverlayRect, cursor);
}

static void CleanupRenderTarget()
{
    if (RenderTarget) { RenderTarget->Release(); RenderTarget = nullptr; }
}

static bool CreateRenderTarget(IDXGISwapChain* swapChain)
{
    ID3D11Texture2D* backBuffer = nullptr;
    if (FAILED(swapChain->GetBuffer(0, __uuidof(ID3D11Texture2D), reinterpret_cast<void**>(&backBuffer)))) return false;
    HRESULT result = Device->CreateRenderTargetView(backBuffer, nullptr, &RenderTarget);
    backBuffer->Release();
    return SUCCEEDED(result);
}

static void CleanupInitialization()
{
    if (OriginalWndProc && GameWindow)
    {
        SetWindowLongPtr(GameWindow, GWLP_WNDPROC, reinterpret_cast<LONG_PTR>(OriginalWndProc));
        OriginalWndProc = nullptr;
    }
    ImGui_ImplDX11_Shutdown();
    ImGui_ImplWin32_Shutdown();
    if (ImGui::GetCurrentContext()) ImGui::DestroyContext();
    CleanupRenderTarget();
    if (Context) { Context->Release(); Context = nullptr; }
    if (Device) { Device->Release(); Device = nullptr; }
    { std::lock_guard<std::mutex> lock(OverlayRectLock); OverlayRectValid = false; }
    GameWindow = nullptr;
    Initialized = false;
}

static bool InitializeOverlay(IDXGISwapChain* swapChain)
{
    if (FAILED(swapChain->GetDevice(__uuidof(ID3D11Device), reinterpret_cast<void**>(&Device)))) return false;
    Device->GetImmediateContext(&Context);
    DXGI_SWAP_CHAIN_DESC description{};
    if (FAILED(swapChain->GetDesc(&description))) { CleanupInitialization(); return false; }
    GameWindow = description.OutputWindow;
    if (!GameWindow || !CreateRenderTarget(swapChain)) { CleanupInitialization(); return false; }
    IMGUI_CHECKVERSION();
    ImGui::CreateContext();
    ImGuiIO& io = ImGui::GetIO();
    io.ConfigFlags |= ImGuiConfigFlags_NoMouseCursorChange;
    io.FontGlobalScale = OverlayConfig::FontScale;
    io.IniFilename = nullptr;
    ImGui::StyleColorsDark();
    if (!ImGui_ImplWin32_Init(GameWindow)) { CleanupInitialization(); return false; }
    if (!ImGui_ImplDX11_Init(Device, Context)) { CleanupInitialization(); return false; }
    OriginalWndProc = reinterpret_cast<WNDPROC>(SetWindowLongPtr(GameWindow, GWLP_WNDPROC, reinterpret_cast<LONG_PTR>(WindowProc)));
    if (!OriginalWndProc) { CleanupInitialization(); return false; }
    Initialized = true;
    return true;
}

static LRESULT CALLBACK WindowProc(HWND hWnd, UINT msg, WPARAM wParam, LPARAM lParam)
{
    if (!OriginalWndProc) return DefWindowProc(hWnd, msg, wParam, lParam);
    if (msg == WM_KEYDOWN && wParam == OverlayConfig::ToggleKey && !(lParam & (1LL << 30)))
    {
        OverlayVisible.store(!OverlayVisible.load());
        return 0;
    }
    if (Initialized && OverlayVisible.load())
    {
        ImGui_ImplWin32_WndProcHandler(hWnd, msg, wParam, lParam);
        bool mouseMessage = msg >= WM_MOUSEFIRST && msg <= WM_MOUSELAST;
        bool mouseButtonDown = msg == WM_LBUTTONDOWN || msg == WM_RBUTTONDOWN || msg == WM_MBUTTONDOWN || msg == WM_XBUTTONDOWN;
        if (mouseMessage)
        {
            bool insideOverlay = IsMouseInsideOverlay(hWnd);
            if (mouseButtonDown && !insideOverlay) OverlayVisible.store(false);
            if (insideOverlay) return 0;
        }
        ImGuiIO& io = ImGui::GetIO();
        if (io.WantCaptureKeyboard && msg >= WM_KEYFIRST && msg <= WM_KEYLAST) return 0;
    }
    return CallWindowProc(OriginalWndProc, hWnd, msg, wParam, lParam);
}

static void DrawOverlay()
{
    if (!OverlayVisible.load()) return;
    ImGui::SetNextWindowSize(ImVec2(OverlayConfig::WindowWidth, OverlayConfig::WindowHeight), ImGuiCond_FirstUseEver);
    bool windowOpen = ImGui::Begin(OverlayConfig::WindowTitle, nullptr, ImGuiWindowFlags_NoCollapse | ImGuiWindowFlags_NoResize);
    UpdateOverlayRect(ImGui::GetWindowPos(), ImGui::GetWindowSize());

    if (windowOpen)
    {
        ImGui::BeginChild("Sidebar", ImVec2(OverlayConfig::SidebarWidth, 0.0f), true);
        if (ImGui::Selectable(OverlayConfig::FeaturesTabName, ActiveTab == 0)) ActiveTab = 0;
        if (ImGui::Selectable(OverlayConfig::AboutTabName, ActiveTab == 1)) ActiveTab = 1;
        ImGui::EndChild();

        ImGui::SameLine();

        ImGui::BeginChild("Content", ImVec2(0.0f, 0.0f), true);
        if (ActiveTab == 0)
        {
            bool energyEnabled = g_unlimitedEnergyEnabled.load();
            if (ImGui::Checkbox(OverlayConfig::EnergyName, &energyEnabled)) g_unlimitedEnergyEnabled.store(energyEnabled);

            bool speedEnabled = g_speedHackEnabled.load();
            if (ImGui::Checkbox(OverlayConfig::SpeedHackName, &speedEnabled)) g_speedHackEnabled.store(speedEnabled);

            ImGui::Spacing();
            ImGui::Separator();
            ImGui::Spacing();

            bool enabled = g_noCooldownEnabled.load();
            if (ImGui::Checkbox(OverlayConfig::NoCooldownName, &enabled))
            {
                g_noCooldownEnabled.store(enabled);
                if (!enabled)
                {
                    g_skillCooldownEnabled.store(false);
                    g_ultimateCooldownEnabled.store(false);
                    g_dashCooldownEnabled.store(false);
                }
            }
            if (g_noCooldownEnabled.load())
            {
                ImGui::Indent(20.0f);
                bool skillEnabled = g_skillCooldownEnabled.load();
                if (ImGui::Checkbox(OverlayConfig::SkillName, &skillEnabled)) g_skillCooldownEnabled.store(skillEnabled);

                bool ultimateEnabled = g_ultimateCooldownEnabled.load();
                if (ImGui::Checkbox(OverlayConfig::UltimateName, &ultimateEnabled)) g_ultimateCooldownEnabled.store(ultimateEnabled);

                bool dashEnabled = g_dashCooldownEnabled.load();
                if (ImGui::Checkbox(OverlayConfig::DashName, &dashEnabled)) g_dashCooldownEnabled.store(dashEnabled);
                ImGui::Unindent(20.0f);
            }
        }
        else if (ActiveTab == 1)
        {
            ImGui::Text("Updated for %s version %s", OverlayConfig::GameName, OverlayConfig::GameVersion);
        }
        ImGui::EndChild();
    }
    ImGui::End();
}

static HRESULT STDMETHODCALLTYPE HookPresent(IDXGISwapChain* swapChain, UINT syncInterval, UINT flags)
{
    {
        std::lock_guard<std::mutex> lock(StateLock);
        if (!Initialized)
        {
            if (!InitializeOverlay(swapChain)) return OriginalPresent(swapChain, syncInterval, flags);
        }
        if (!OverlayVisible.load()) return OriginalPresent(swapChain, syncInterval, flags);
        ImGui_ImplDX11_NewFrame();
        ImGui_ImplWin32_NewFrame();
        ImGui::NewFrame();
        DrawOverlay();
        ImGui::Render();
        Context->OMSetRenderTargets(1, &RenderTarget, nullptr);
        ImGui_ImplDX11_RenderDrawData(ImGui::GetDrawData());
    }
    return OriginalPresent(swapChain, syncInterval, flags);
}

static HRESULT STDMETHODCALLTYPE HookResizeBuffers(IDXGISwapChain* swapChain, UINT bufferCount, UINT width, UINT height, DXGI_FORMAT newFormat, UINT swapChainFlags)
{
    std::lock_guard<std::mutex> lock(StateLock);
    if (Initialized)
    {
        CleanupRenderTarget();
        ImGui_ImplDX11_InvalidateDeviceObjects();
    }
    HRESULT result = OriginalResizeBuffers(swapChain, bufferCount, width, height, newFormat, swapChainFlags);
    if (Initialized)
    {
        if (CreateRenderTarget(swapChain)) ImGui_ImplDX11_CreateDeviceObjects();
    }
    return result;
}

bool GetD3D11Targets(void** presentTarget, void** resizeTarget)
{
    if (!presentTarget || !resizeTarget) return false;
    *presentTarget = nullptr;
    *resizeTarget = nullptr;

    WNDCLASSEX windowClass{};
    windowClass.cbSize = sizeof(WNDCLASSEX);
    windowClass.lpfnWndProc = DefWindowProc;
    windowClass.hInstance = GetModuleHandle(nullptr);
    windowClass.lpszClassName = TEXT("D3D11DummyWindow");

    if (!RegisterClassEx(&windowClass) && GetLastError() != ERROR_CLASS_ALREADY_EXISTS) return false;

    HWND dummyWindow = CreateWindowEx(0, windowClass.lpszClassName, TEXT("D3D11DummyWindow"), WS_OVERLAPPEDWINDOW, 0, 0, 100, 100, nullptr, nullptr, windowClass.hInstance, nullptr);
    if (!dummyWindow) return false;

    DXGI_SWAP_CHAIN_DESC swapDesc{};
    swapDesc.BufferCount = 1;
    swapDesc.BufferDesc.Format = DXGI_FORMAT_R8G8B8A8_UNORM;
    swapDesc.BufferUsage = DXGI_USAGE_RENDER_TARGET_OUTPUT;
    swapDesc.OutputWindow = dummyWindow;
    swapDesc.SampleDesc.Count = 1;
    swapDesc.Windowed = TRUE;
    swapDesc.SwapEffect = DXGI_SWAP_EFFECT_DISCARD;

    D3D_FEATURE_LEVEL featureLevel{};
    const D3D_FEATURE_LEVEL levels[] = { D3D_FEATURE_LEVEL_11_0, D3D_FEATURE_LEVEL_10_0 };

    IDXGISwapChain* swapChain = nullptr;
    ID3D11Device* device = nullptr;
    ID3D11DeviceContext* context = nullptr;

    HRESULT result = D3D11CreateDeviceAndSwapChain(nullptr, D3D_DRIVER_TYPE_HARDWARE, nullptr, 0, levels, ARRAYSIZE(levels), D3D11_SDK_VERSION, &swapDesc, &swapChain, &device, &featureLevel, &context);
    if (FAILED(result))
    {
        result = D3D11CreateDeviceAndSwapChain(nullptr, D3D_DRIVER_TYPE_WARP, nullptr, 0, levels, ARRAYSIZE(levels), D3D11_SDK_VERSION, &swapDesc, &swapChain, &device, &featureLevel, &context);
    }

    bool success = false;
    if (SUCCEEDED(result) && swapChain)
    {
        void** vtable = *reinterpret_cast<void***>(swapChain);
        *presentTarget = vtable[8];
        *resizeTarget = vtable[13];
        success = *presentTarget && *resizeTarget;
    }

    if (context) context->Release();
    if (device) device->Release();
    if (swapChain) swapChain->Release();
    DestroyWindow(dummyWindow);
    UnregisterClass(windowClass.lpszClassName, windowClass.hInstance);
    return success;
}

bool InstallOverlayHooks()
{
    void* presentTarget = nullptr;
    void* resizeTarget = nullptr;
    if (!GetD3D11Targets(&presentTarget, &resizeTarget)) return false;

    if (MH_CreateHook(presentTarget, reinterpret_cast<LPVOID>(&HookPresent), reinterpret_cast<LPVOID*>(&OriginalPresent)) != MH_OK) return false;
    if (MH_CreateHook(resizeTarget, reinterpret_cast<LPVOID>(&HookResizeBuffers), reinterpret_cast<LPVOID*>(&OriginalResizeBuffers)) != MH_OK) return false;

    if (MH_EnableHook(presentTarget) != MH_OK) return false;
    if (MH_EnableHook(resizeTarget) != MH_OK) return false;

    return true;
}

Overwriting overlay.cpp


**Cell 4: Cooldown**

In [ ]:
%%writefile cooldown.cpp


#include "shared.h"
#include "MinHook.h"

#include <cstdint>
#include <cstddef>
#include <cstring>
#include <cstdio>
#include <cstdarg>
#include <string>
#include <unordered_set>
#include <atomic>
#include <unordered_map>
#include <mutex>

struct Il2CppDomain;
struct Il2CppThread;
struct Il2CppAssembly;
struct Il2CppImage;
struct Il2CppClass;
struct Il2CppField;
struct MethodInfo;

using DomainGetFn = Il2CppDomain* (*)();
using ThreadAttachFn = Il2CppThread* (*)(Il2CppDomain*);
using DomainGetAssembliesFn = const Il2CppAssembly** (*)(const Il2CppDomain*, size_t*);
using AssemblyGetImageFn = const Il2CppImage* (*)(const Il2CppAssembly*);
using ImageGetClassCountFn = size_t (*)(const Il2CppImage*);
using ImageGetClassFn = Il2CppClass* (*)(const Il2CppImage*, size_t);
using ClassGetNameFn = const char* (*)(Il2CppClass*);
using ClassGetParentFn = Il2CppClass* (*)(Il2CppClass*);
using ClassGetMethodFn = const MethodInfo* (*)(Il2CppClass*, const char*, int);
using ClassGetFieldFn = Il2CppField* (*)(Il2CppClass*, const char*);
using FieldGetOffsetFn = size_t (*)(Il2CppField*);
using ObjectGetClassFn = Il2CppClass* (*)(void*);
using ObjectUnboxFn = void* (*)(void*);
using RuntimeInvokeFn = void* (*)(const MethodInfo*, void*, void**, void**);
using LuaPCallFn = int (*)(void*, int, int, int);

static HMODULE g_gameAssembly = nullptr;
static HMODULE g_xlua = nullptr;

static DomainGetFn g_domainGet = nullptr;
static ThreadAttachFn g_threadAttach = nullptr;
static DomainGetAssembliesFn g_domainGetAssemblies = nullptr;
static AssemblyGetImageFn g_assemblyGetImage = nullptr;
static ImageGetClassCountFn g_imageGetClassCount = nullptr;
static ImageGetClassFn g_imageGetClass = nullptr;
static ClassGetNameFn g_classGetName = nullptr;
static ClassGetParentFn g_classGetParent = nullptr;
static ClassGetMethodFn g_classGetMethod = nullptr;
static ClassGetFieldFn g_classGetField = nullptr;
static FieldGetOffsetFn g_fieldGetOffset = nullptr;
static ObjectGetClassFn g_objectGetClass = nullptr;
static ObjectUnboxFn g_objectUnbox = nullptr;
static RuntimeInvokeFn g_runtimeInvoke = nullptr;
static LuaPCallFn g_luaPCallOriginal = nullptr;

static std::atomic<int> g_cooldownStatus{0};
static std::atomic_flag g_initializationBusy = ATOMIC_FLAG_INIT;
static std::atomic_flag g_cooldownBusy = ATOMIC_FLAG_INIT;
static thread_local bool g_il2cppThreadAttached = false;

struct OriginalMovementSpeeds {
    int64_t runSpeed;
    int64_t walkSpeed;
    int64_t fastRunSpeed;
    void* actorInfo;
};

static std::unordered_map<void*, OriginalMovementSpeeds> g_originalMovementSpeeds;
static std::mutex g_speedMutex;

static constexpr char ACTOR_CLASS[] = "PlayerAdventureActor";
static constexpr char API_CLASS[] = "ILRuntimeAPI";
static constexpr char ACTOR_COOLDOWN_FIELD[] = "playerSkillCd";

// TrueSync fixed point formats: FP has 32 fractional bits, FDP has 24
static constexpr int64_t FP_ONE = 1LL << 32;
static constexpr int64_t FDP_ONE = 1LL << 24;
static constexpr int64_t TARGET_SPEED = 25 * FP_ONE;

// Skill slots passed to ResetSkillCD
static constexpr int32_t SKILL_ID_DASH = 1;
static constexpr int32_t SKILL_ID_SKILL = 2;
static constexpr int32_t SKILL_ID_ULTIMATE = 4;

static constexpr ULONGLONG COOLDOWN_TICK_MS = 0x77;

// IL2CPP 64-bit layouts: List<T> { header 0x10, _items, _size }, T[] { header 0x10, bounds, length, data }
static constexpr uintptr_t LIST_ITEMS_OFFSET = 0x10;
static constexpr uintptr_t LIST_SIZE_OFFSET = 0x18;
static constexpr uintptr_t ARRAY_DATA_OFFSET = 0x20;
static constexpr uint32_t MAX_ACTOR_COUNT = 0x1000;

// GetTotalEnergy results outside this range mean the read is wrong, not that energy is huge
static constexpr int32_t MIN_VALID_ENERGY = 1;
static constexpr int32_t MAX_VALID_ENERGY = 1000;

enum CooldownStatus {
    COOLDOWN_WAITING = 0,
    COOLDOWN_READY = 1,
    COOLDOWN_ERROR_GAMEASSEMBLY = -1,
    COOLDOWN_ERROR_EXPORTS = -2,
    COOLDOWN_ERROR_THREAD_ATTACH = -4,
    COOLDOWN_ERROR_XLUA = -5,
    COOLDOWN_ERROR_LUA_EXPORT = -6,
    COOLDOWN_ERROR_HOOK_CREATE = -7,
    COOLDOWN_ERROR_HOOK_ENABLE = -8
};

// Debug
static void LogOnce(const char* format, ...)
{
    char message[256];

    va_list args;
    va_start(args, format);
    std::vsnprintf(message, sizeof(message), format, args);
    va_end(args);

    static std::mutex mutex;
    static std::unordered_set<std::string> seen;

    std::lock_guard<std::mutex> lock(mutex);

    if (seen.size() >= 64 || !seen.insert(message).second)
        return;

    std::string line = std::string("[cooldown] ") + message + "\n";
    OutputDebugStringA(line.c_str());
}

template <typename T>
static T ResolveExport(const char* name)
{
    return reinterpret_cast<T>(GetProcAddress(g_gameAssembly, name));
}

static bool ResolveIL2CPP()
{
    g_gameAssembly = GetModuleHandleA("GameAssembly.dll");
    if (!g_gameAssembly) return false;

    g_domainGet = ResolveExport<DomainGetFn>("il2cpp_domain_get");
    g_threadAttach = ResolveExport<ThreadAttachFn>("il2cpp_thread_attach");
    g_domainGetAssemblies = ResolveExport<DomainGetAssembliesFn>("il2cpp_domain_get_assemblies");
    g_assemblyGetImage = ResolveExport<AssemblyGetImageFn>("il2cpp_assembly_get_image");
    g_imageGetClassCount = ResolveExport<ImageGetClassCountFn>("il2cpp_image_get_class_count");
    g_imageGetClass = ResolveExport<ImageGetClassFn>("il2cpp_image_get_class");
    g_classGetName = ResolveExport<ClassGetNameFn>("il2cpp_class_get_name");
    g_classGetParent = ResolveExport<ClassGetParentFn>("il2cpp_class_get_parent");
    g_classGetMethod = ResolveExport<ClassGetMethodFn>("il2cpp_class_get_method_from_name");
    g_classGetField = ResolveExport<ClassGetFieldFn>("il2cpp_class_get_field_from_name");
    g_fieldGetOffset = ResolveExport<FieldGetOffsetFn>("il2cpp_field_get_offset");
    g_objectGetClass = ResolveExport<ObjectGetClassFn>("il2cpp_object_get_class");
    g_objectUnbox = ResolveExport<ObjectUnboxFn>("il2cpp_object_unbox");
    g_runtimeInvoke = ResolveExport<RuntimeInvokeFn>("il2cpp_runtime_invoke");

    return g_domainGet &&
           g_threadAttach &&
           g_domainGetAssemblies &&
           g_assemblyGetImage &&
           g_imageGetClassCount &&
           g_imageGetClass &&
           g_classGetName &&
           g_classGetParent &&
           g_classGetMethod &&
           g_classGetField &&
           g_fieldGetOffset &&
           g_objectGetClass &&
           g_objectUnbox &&
           g_runtimeInvoke;
}

static bool EnsureIL2CPPThreadAttached()
{
    if (g_il2cppThreadAttached) return true;
    if (!g_domainGet || !g_threadAttach) return false;

    Il2CppDomain* domain = g_domainGet();
    if (!domain) return false;

    Il2CppThread* thread = g_threadAttach(domain);
    if (!thread) return false;

    g_il2cppThreadAttached = true;
    return true;
}

static Il2CppClass* FindClassByName(
    Il2CppDomain* domain,
    const char* targetName)
{
    size_t assemblyCount = 0;

    const Il2CppAssembly** assemblies =
        g_domainGetAssemblies(domain, &assemblyCount);

    if (!assemblies || assemblyCount == 0)
        return nullptr;

    Il2CppClass* found = nullptr;
    size_t matches = 0;

    for (size_t i = 0; i < assemblyCount; ++i) {
        if (!assemblies[i]) continue;

        const Il2CppImage* image =
            g_assemblyGetImage(assemblies[i]);

        if (!image) continue;

        size_t classCount = g_imageGetClassCount(image);

        for (size_t j = 0; j < classCount; ++j) {
            Il2CppClass* klass = g_imageGetClass(image, j);
            if (!klass) continue;

            const char* name = g_classGetName(klass);

            if (!name || std::strcmp(name, targetName) != 0)
                continue;

            found = klass;
            ++matches;
        }
    }

    return matches == 1 ? found : nullptr;
}

static const MethodInfo* FindMethodInHierarchy(
    Il2CppClass* klass,
    const char* methodName,
    int parameterCount)
{
    for (Il2CppClass* current = klass;
         current != nullptr;
         current = g_classGetParent(current)) {

        const MethodInfo* method =
            g_classGetMethod(current, methodName, parameterCount);

        if (method) return method;
    }

    return nullptr;
}

static bool GetFieldOffset(
    Il2CppClass* klass,
    const char* fieldName,
    size_t* result)
{
    if (!klass || !result)
        return false;

    for (Il2CppClass* current = klass;
         current != nullptr;
         current = g_classGetParent(current)) {

        Il2CppField* field =
            g_classGetField(current, fieldName);

        if (field) {
            *result = g_fieldGetOffset(field);
            return true;
        }
    }

    return false;
}

static bool InvokeMethod(
    const MethodInfo* method,
    void* instance,
    void** arguments,
    void** returnedObject = nullptr)
{
    if (!method) return false;

    void* exception = nullptr;

    void* result = g_runtimeInvoke(
        method,
        instance,
        arguments,
        &exception
    );

    if (exception) return false;

    if (returnedObject)
        *returnedObject = result;

    return true;
}

static bool GetActorFPPropertyRaw(
    void* actorInfo,
    const char* getterName,
    int64_t* rawValue)
{
    if (!actorInfo || !getterName || !rawValue)
        return false;

    Il2CppClass* infoClass =
        g_objectGetClass(actorInfo);

    if (!infoClass)
        return false;

    const MethodInfo* getter =
        FindMethodInHierarchy(
            infoClass,
            getterName,
            0
        );

    if (!getter)
        return false;

    void* boxedResult = nullptr;

    if (!InvokeMethod(
        getter,
        actorInfo,
        nullptr,
        &boxedResult))
        return false;

    if (!boxedResult)
        return false;

    void* unboxedResult =
        g_objectUnbox(boxedResult);

    if (!unboxedResult)
        return false;

    std::memcpy(
        rawValue,
        unboxedResult,
        sizeof(int64_t)
    );

    return true;
}

static bool SetActorFPPropertyRaw(
    void* actorInfo,
    const char* setterName,
    int64_t rawValue)
{
    if (!actorInfo || !setterName)
        return false;

    Il2CppClass* infoClass =
        g_objectGetClass(actorInfo);

    if (!infoClass)
        return false;

    const MethodInfo* setter =
        FindMethodInHierarchy(
            infoClass,
            setterName,
            1
        );

    if (!setter)
        return false;

    void* arguments[] = { &rawValue };

    return InvokeMethod(
        setter,
        actorInfo,
        arguments
    );
}

// Method pointers are stable for the life of the process, so resolve each one once per class.
// A method that is not found stays null and is retried, never cached as missing.
struct CooldownMethods {
    Il2CppClass* klass = nullptr;
    const MethodInfo* resetSkill = nullptr;
    const MethodInfo* getTotalEnergy = nullptr;
    const MethodInfo* setEnergy = nullptr;
};

static const CooldownMethods& ResolveCooldownMethods(Il2CppClass* klass)
{
    static CooldownMethods cache;

    if (cache.klass != klass)
        cache = CooldownMethods{ klass };

    if (!cache.resetSkill)
        cache.resetSkill = FindMethodInHierarchy(klass, "ResetSkillCD", 1);
    if (!cache.getTotalEnergy)
        cache.getTotalEnergy = FindMethodInHierarchy(klass, "GetTotalEnergy", 0);
    if (!cache.setEnergy)
        cache.setEnergy = FindMethodInHierarchy(klass, "SetEnergy", 1);

    return cache;
}

static void ResetCooldown(
    const CooldownMethods& methods,
    void* cooldownObject,
    int32_t skillId)
{
    if (!methods.resetSkill) {
        LogOnce("ResetSkillCD not found");
        return;
    }

    int32_t skillIndex = skillId;
    void* arguments[] = { &skillIndex };

    if (!InvokeMethod(methods.resetSkill, cooldownObject, arguments))
        LogOnce("ResetSkillCD threw for skill %d", (int)skillId);
}

static void ProcessUnlimitedEnergy(
    const CooldownMethods& methods,
    void* cooldownObject)
{
    if (!g_unlimitedEnergyEnabled.load())
        return;

    if (!methods.getTotalEnergy || !methods.setEnergy) {
        LogOnce("GetTotalEnergy or SetEnergy not found");
        return;
    }

    void* boxedTotal = nullptr;

    if (!InvokeMethod(methods.getTotalEnergy, cooldownObject, nullptr, &boxedTotal) ||
        !boxedTotal) {
        LogOnce("GetTotalEnergy failed");
        return;
    }

    const int32_t* total = static_cast<const int32_t*>(g_objectUnbox(boxedTotal));

    if (!total) {
        LogOnce("GetTotalEnergy result could not be unboxed");
        return;
    }

    int32_t maxEnergy = *total;

    if (maxEnergy < MIN_VALID_ENERGY || maxEnergy > MAX_VALID_ENERGY) {
        LogOnce("GetTotalEnergy returned %d, expected %d to %d",
            (int)maxEnergy, (int)MIN_VALID_ENERGY, (int)MAX_VALID_ENERGY);
        return;
    }

    int64_t targetEnergyFDP = static_cast<int64_t>(maxEnergy) * FDP_ONE;
    void* arguments[] = { &targetEnergyFDP };

    if (!InvokeMethod(methods.setEnergy, cooldownObject, arguments))
        LogOnce("SetEnergy threw");
}

// Direct pointer to an FP field on ActorInfo, for fields without a usable property
static int64_t* GetInfoFieldPtr(void* actorInfo, const char* fieldName)
{
    size_t offset = 0;

    if (!GetFieldOffset(g_objectGetClass(actorInfo), fieldName, &offset))
        return nullptr;

    return reinterpret_cast<int64_t*>(
        reinterpret_cast<uintptr_t>(actorInfo) + offset
    );
}

// The actor caches its movement speed and only re-reads actorInfo on certain events like a dash.
// Call the game's own refresh methods once so the new values apply without needing to dash.
static void RefreshActorMovement(void* actor, Il2CppClass* actorClass)
{
    bool isInit = false;
    void* initArgs[] = { &isInit };

    if (!InvokeMethod(
        FindMethodInHierarchy(actorClass, "RefreshMoveSpeed", 1), actor, initArgs))
        LogOnce("RefreshMoveSpeed not found or threw");
}

static void ProcessSpeedHack(void* actor)
{
    if (!actor)
        return;

    Il2CppClass* actorClass =
        g_objectGetClass(actor);

    if (!actorClass)
        return;

    size_t infoOffset = 0;
    void* actorInfo = nullptr;

    if (GetFieldOffset(actorClass, "actorInfo", &infoOffset) ||
        GetFieldOffset(actorClass, "m_ActorInfo", &infoOffset)) {

        actorInfo =
            *reinterpret_cast<void**>(
                reinterpret_cast<uintptr_t>(actor) + infoOffset
            );
    }
    else {
        const MethodInfo* getInfo =
            FindMethodInHierarchy(
                actorClass,
                "get_actorInfo",
                0
            );

        if (getInfo) {
            InvokeMethod(
                getInfo,
                actor,
                nullptr,
                &actorInfo
            );
        }
    }

    if (!actorInfo)
        return;

    std::lock_guard<std::mutex> lock(g_speedMutex);

    bool enabled = g_speedHackEnabled.load();
    bool firstApply = false;

    auto it = g_originalMovementSpeeds.find(actor);

    // Drop instead of restoring
    if (it != g_originalMovementSpeeds.end() && it->second.actorInfo != actorInfo) {
        g_originalMovementSpeeds.erase(it);
        it = g_originalMovementSpeeds.end();
    }

    if (enabled) {
        if (it == g_originalMovementSpeeds.end()) {
            OriginalMovementSpeeds original{};

            // get_runSpeed returns fastRunSpeed while fastSpeedMode is on, so read the field
            int64_t* runField = GetInfoFieldPtr(actorInfo, "_runSpeed");
            if (!runField) {
                LogOnce("ActorInfo._runSpeed not found");
                return;
            }

            original.runSpeed = *runField;

            if (!GetActorFPPropertyRaw(
                actorInfo,
                "get_walkSpeed",
                &original.walkSpeed)) {
                LogOnce("ActorInfo.get_walkSpeed failed");
                return;
            }

            int64_t* fastRun = GetInfoFieldPtr(actorInfo, "fastRunSpeed");
            if (!fastRun) {
                LogOnce("ActorInfo.fastRunSpeed not found");
                return;
            }

            original.fastRunSpeed = *fastRun;
            original.actorInfo = actorInfo;

            it = g_originalMovementSpeeds.emplace(
                actor,
                original
            ).first;

            firstApply = true;
        }

        SetActorFPPropertyRaw(
            actorInfo,
            "set_runSpeed",
            TARGET_SPEED
        );

        SetActorFPPropertyRaw(
            actorInfo,
            "set_walkSpeed",
            TARGET_SPEED
        );

        // Sprinting uses fastRunSpeed
        if (int64_t* fastRun = GetInfoFieldPtr(actorInfo, "fastRunSpeed"))
            *fastRun = TARGET_SPEED;

        if (firstApply)
            RefreshActorMovement(actor, actorClass);
    }
    else {
        if (it != g_originalMovementSpeeds.end()) {
            SetActorFPPropertyRaw(
                actorInfo,
                "set_runSpeed",
                it->second.runSpeed
            );

            SetActorFPPropertyRaw(
                actorInfo,
                "set_walkSpeed",
                it->second.walkSpeed
            );

            if (int64_t* fastRun = GetInfoFieldPtr(actorInfo, "fastRunSpeed"))
                *fastRun = it->second.fastRunSpeed;

            RefreshActorMovement(actor, actorClass);

            g_originalMovementSpeeds.erase(it);
        }
    }
}

static const MethodInfo* g_getActors = nullptr;

static bool GetCooldownOffset(Il2CppClass* actorClass, size_t* offset)
{
    static Il2CppClass* cachedClass = nullptr;
    static size_t cachedOffset = 0;

    if (cachedClass != actorClass) {
        if (!GetFieldOffset(actorClass, ACTOR_COOLDOWN_FIELD, &cachedOffset)) {
            LogOnce("field %s not found on %s", ACTOR_COOLDOWN_FIELD, g_classGetName(actorClass));
            return false;
        }

        cachedClass = actorClass;
    }

    *offset = cachedOffset;
    return true;
}

static void ProcessActors()
{
    if (!g_getActors) {
        Il2CppDomain* domain = g_domainGet();

        if (!domain) {
            LogOnce("il2cpp domain not available");
            return;
        }

        Il2CppClass* apiClass = FindClassByName(domain, API_CLASS);

        if (!apiClass) {
            LogOnce("class %s not found or not unique", API_CLASS);
            return;
        }

        g_getActors = FindMethodInHierarchy(apiClass, "GetAllPlayerActorAndMinion", 1);

        if (!g_getActors) {
            LogOnce("GetAllPlayerActorAndMinion not found");
            return;
        }
    }

    bool includeMinion = false;
    void* arguments[] = { &includeMinion };
    void* actors = nullptr;

    if (!InvokeMethod(g_getActors, nullptr, arguments, &actors)) {
        LogOnce("GetAllPlayerActorAndMinion threw");
        return;
    }

    if (!actors)
        return;

    uint32_t count =
        *reinterpret_cast<uint32_t*>(
            reinterpret_cast<uintptr_t>(actors) + LIST_SIZE_OFFSET
        );

    void* elements =
        *reinterpret_cast<void**>(
            reinterpret_cast<uintptr_t>(actors) + LIST_ITEMS_OFFSET
        );

    if (!elements || count > MAX_ACTOR_COUNT)
        return;

    for (uint32_t i = 0; i < count; ++i) {
        void* actor =
            *reinterpret_cast<void**>(
                reinterpret_cast<uintptr_t>(elements) +
                ARRAY_DATA_OFFSET +
                (static_cast<uintptr_t>(i) * sizeof(void*))
            );

        if (!actor)
            continue;

        Il2CppClass* actorClass =
            g_objectGetClass(actor);

        if (!actorClass)
            continue;

        const char* className =
            g_classGetName(actorClass);

        if (!className ||
            !std::strstr(className, ACTOR_CLASS))
            continue;

        ProcessSpeedHack(actor);

        size_t cooldownOffset = 0;

        if (!GetCooldownOffset(actorClass, &cooldownOffset))
            continue;

        void* cooldownObject =
            *reinterpret_cast<void**>(
                reinterpret_cast<uintptr_t>(actor) +
                cooldownOffset
            );

        if (!cooldownObject)
            continue;

        Il2CppClass* cooldownClass =
            g_objectGetClass(cooldownObject);

        if (!cooldownClass)
            continue;

        const CooldownMethods& methods =
            ResolveCooldownMethods(cooldownClass);

        if (g_skillCooldownEnabled.load())
            ResetCooldown(methods, cooldownObject, SKILL_ID_SKILL);

        if (g_ultimateCooldownEnabled.load())
            ResetCooldown(methods, cooldownObject, SKILL_ID_ULTIMATE);

        if (g_dashCooldownEnabled.load())
            ResetCooldown(methods, cooldownObject, SKILL_ID_DASH);

        ProcessUnlimitedEnergy(methods, cooldownObject);
    }
}

static void ProcessCooldownActors()
{
    if (g_cooldownBusy.test_and_set())
        return;

    ProcessActors();

    g_cooldownBusy.clear();
}

static int InitializeIL2CPP()
{
    if (!ResolveIL2CPP())
        return g_gameAssembly ? COOLDOWN_ERROR_EXPORTS : COOLDOWN_ERROR_GAMEASSEMBLY;

    if (!EnsureIL2CPPThreadAttached())
        return COOLDOWN_ERROR_THREAD_ATTACH;

    return COOLDOWN_READY;
}

static void InitializeIL2CPPRuntimeOnLuaThread()
{
    if (g_cooldownStatus.load() == COOLDOWN_READY)
        return;

    if (g_initializationBusy.test_and_set())
        return;

    int status = InitializeIL2CPP();
    g_cooldownStatus.store(status);

    if (status != COOLDOWN_READY)
        LogOnce("il2cpp init failed, status %d", status);

    g_initializationBusy.clear();
}

static int __fastcall LuaPCall_Hook(
    void* state,
    int nargs,
    int nresults,
    int errfunc)
{
    static ULONGLONG lastCooldownTick = 0;

    InitializeIL2CPPRuntimeOnLuaThread();

    if (g_cooldownStatus.load() == COOLDOWN_READY) {
        if (EnsureIL2CPPThreadAttached()) {
            ULONGLONG now = GetTickCount64();

            if (now - lastCooldownTick > COOLDOWN_TICK_MS) {
                lastCooldownTick = now;
                ProcessCooldownActors();
            }
        }
    }

    LuaPCallFn original = g_luaPCallOriginal;

    if (!original)
        return 0;

    return original(
        state,
        nargs,
        nresults,
        errfunc
    );
}

static bool FailInstall(CooldownStatus status, const char* reason)
{
    g_cooldownStatus.store(status);
    LogOnce("hook install failed: %s (status %d)", reason, (int)status);
    return false;
}

bool InstallCooldownHook()
{
    for (int i = 0; i < 600; ++i) {
        g_xlua = GetModuleHandleA("xlua.dll");

        if (g_xlua)
            break;

        Sleep(100);
    }

    if (!g_xlua)
        return FailInstall(COOLDOWN_ERROR_XLUA, "xlua.dll not loaded");

    FARPROC luaPCall =
        GetProcAddress(g_xlua, "lua_pcall");

    if (!luaPCall)
        return FailInstall(COOLDOWN_ERROR_LUA_EXPORT, "lua_pcall export not found");

    if (MH_CreateHook(
        reinterpret_cast<LPVOID>(luaPCall),
        reinterpret_cast<LPVOID>(&LuaPCall_Hook),
        reinterpret_cast<LPVOID*>(&g_luaPCallOriginal)
    ) != MH_OK)
        return FailInstall(COOLDOWN_ERROR_HOOK_CREATE, "MH_CreateHook failed");

    if (MH_EnableHook(
        reinterpret_cast<LPVOID>(luaPCall)
    ) != MH_OK) {

        MH_RemoveHook(reinterpret_cast<LPVOID>(luaPCall));
        g_luaPCallOriginal = nullptr;

        return FailInstall(COOLDOWN_ERROR_HOOK_ENABLE, "MH_EnableHook failed");
    }

    return true;
}


Overwriting cooldown.cpp


**CELL 5: Main**

In [ ]:
%%writefile main.cpp

#include "shared.h"
#include "MinHook.h"

#include <atomic>

std::atomic<bool> g_noCooldownEnabled{ false };
std::atomic<bool> g_skillCooldownEnabled{ false };
std::atomic<bool> g_ultimateCooldownEnabled{ false };
std::atomic<bool> g_dashCooldownEnabled{ false };
std::atomic<bool> g_unlimitedEnergyEnabled{ false };
std::atomic<bool> g_speedHackEnabled{ false };

static DWORD WINAPI InitializeThread(void*)
{
    MH_STATUS status = MH_Initialize();

    if (status != MH_OK &&
        status != MH_ERROR_ALREADY_INITIALIZED)
    {
        return 0;
    }

    InstallOverlayHooks();
    InstallCooldownHook();

    return 0;
}

BOOL WINAPI DllMain(
    HINSTANCE instance,
    DWORD reason,
    LPVOID
)
{
    if (reason == DLL_PROCESS_ATTACH)
    {
        DisableThreadLibraryCalls(instance);

        HANDLE thread = CreateThread(
            nullptr,
            0,
            InitializeThread,
            nullptr,
            0,
            nullptr
        );

        if (thread)
        {
            CloseHandle(thread);
        }
    }

    return TRUE;
}

Overwriting main.cpp


**CELL 6: Compile the DLL**

In [ ]:
%%bash
set -e

MINHOOK_DIR="/content/minhook"
IMGUI_DIR="/content/imgui"

echo "Compiling MinHook..."

x86_64-w64-mingw32-gcc -O2 \
    -I"$MINHOOK_DIR/include" \
    -I"$MINHOOK_DIR/src" \
    -c "$MINHOOK_DIR/src/buffer.c" -o buffer.o

x86_64-w64-mingw32-gcc -O2 \
    -I"$MINHOOK_DIR/include" \
    -I"$MINHOOK_DIR/src" \
    -c "$MINHOOK_DIR/src/hook.c" -o hook.o

x86_64-w64-mingw32-gcc -O2 \
    -I"$MINHOOK_DIR/include" \
    -I"$MINHOOK_DIR/src" \
    -c "$MINHOOK_DIR/src/trampoline.c" -o trampoline.o

x86_64-w64-mingw32-gcc -O2 \
    -I"$MINHOOK_DIR/include" \
    -I"$MINHOOK_DIR/src" \
    -c "$MINHOOK_DIR/src/hde/hde64.c" -o hde64.o

echo "Compiling project..."

x86_64-w64-mingw32-g++ \
    -std=c++17 -O2 -shared \
    -static-libgcc -static-libstdc++ \
    -Wl,--gc-sections -s \
    -I/content \
    -I"$MINHOOK_DIR/include" \
    -I"$IMGUI_DIR" \
    -I"$IMGUI_DIR/backends" \
    main.cpp overlay.cpp cooldown.cpp \
    "$IMGUI_DIR/imgui.cpp" \
    "$IMGUI_DIR/imgui_draw.cpp" \
    "$IMGUI_DIR/imgui_tables.cpp" \
    "$IMGUI_DIR/imgui_widgets.cpp" \
    "$IMGUI_DIR/backends/imgui_impl_win32.cpp" \
    "$IMGUI_DIR/backends/imgui_impl_dx11.cpp" \
    buffer.o hook.o trampoline.o hde64.o \
    -ld3d11 -ldxgi -ld3dcompiler -ldwmapi -lgdi32 -limm32 \
    -o MG_cheat.dll

echo "Build completed."
ls -lh MG_cheat.dll


Compiling MinHook...
Compiling project...
Build completed.
-rwxr-xr-x 1 root root 1.1M Nov 16 05:40 MG_cheat.dll


**Download**

In [ ]:
from google.colab import files
files.download("MG_cheat.dll")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>